# 04 — RAG Knowledge Base Setup

Build the **ChromaDB** vector database that powers **Agent 2** (Clinical Evidence Retriever).

| Step | What happens |
|------|--------------|
| 1 | Define 27 clinical paper snippets covering PCOS diagnostic criteria, hormone ranges, and guidelines |
| 2 | Initialise a persistent Chroma collection (`clinical_papers`) |
| 3 | Embed and store every document |
| 4 | Run test queries to verify semantic retrieval works |

**Output →** `knowledge_base/chroma_db/` (used by `src/rag_system.py`)

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CHROMA_DIR = ROOT / "knowledge_base" / "chroma_db"
COLLECTION = "clinical_papers"
print(f"Project root : {ROOT}")
print(f"Chroma path  : {CHROMA_DIR}")

Project root : /Users/apple/Desktop/GradStudy/SYSEN5381/PCOSense
Chroma path  : /Users/apple/Desktop/GradStudy/SYSEN5381/PCOSense/knowledge_base/chroma_db


## 1 — Define Clinical Papers

27 curated text snippets drawn from key PCOS literature:  
Rotterdam Criteria, ESHRE/ASRM guidelines, hormone thresholds, metabolic features, and more.

In [2]:
CLINICAL_PAPERS = [
    # ── Rotterdam Criteria & Consensus ─────────────────────────────────
    {
        "id": "rotterdam_2003",
        "title": "Rotterdam ESHRE/ASRM-Sponsored PCOS Consensus Workshop 2003",
        "year": "2004",
        "source": "Human Reproduction",
        "document": (
            "The Rotterdam criteria for PCOS diagnosis require at least two of three features: "
            "(1) oligo-ovulation or anovulation, (2) clinical and/or biochemical signs of "
            "hyperandrogenism, and (3) polycystic ovarian morphology on ultrasound. Other "
            "aetiologies must be excluded, including congenital adrenal hyperplasia, "
            "androgen-secreting tumours, and Cushing syndrome. The consensus broadened the "
            "1990 NIH criteria by adding ovarian morphology as a diagnostic pillar. This "
            "allows identification of additional phenotypes including normo-androgenic PCOS "
            "with polycystic ovaries and oligomenorrhea, and ovulatory PCOS with "
            "hyperandrogenism and polycystic ovaries."
        ),
    },
    {
        "id": "rotterdam_phenotypes",
        "title": "PCOS Phenotypes Under the Rotterdam Criteria",
        "year": "2012",
        "source": "Fertility and Sterility",
        "document": (
            "The Rotterdam criteria define four distinct PCOS phenotypes: "
            "Phenotype A (classic) — hyperandrogenism + oligo-anovulation + polycystic ovaries; "
            "Phenotype B — hyperandrogenism + oligo-anovulation without polycystic ovaries; "
            "Phenotype C — hyperandrogenism + polycystic ovaries with regular ovulation; "
            "Phenotype D — oligo-anovulation + polycystic ovaries without hyperandrogenism. "
            "Phenotypes A and B carry the highest metabolic risk, including insulin resistance, "
            "dyslipidaemia, and increased cardiovascular risk. Phenotype D is the mildest form. "
            "Phenotype prevalence varies by population, with A being most common (60-70% of cases)."
        ),
    },
    {
        "id": "rotterdam_application",
        "title": "Applying the Rotterdam Criteria in Clinical Practice",
        "year": "2016",
        "source": "Journal of Clinical Endocrinology & Metabolism",
        "document": (
            "Clinical application of Rotterdam criteria requires systematic evaluation. "
            "Oligo-anovulation is defined as menstrual cycles longer than 35 days or fewer "
            "than 8 cycles per year. Biochemical hyperandrogenism is assessed via total "
            "testosterone, free testosterone, or free androgen index (FAI). Clinical "
            "hyperandrogenism includes hirsutism (modified Ferriman-Gallwey score ≥4-6), "
            "acne, or androgenic alopecia. Polycystic ovarian morphology requires ≥12 "
            "follicles measuring 2-9 mm in diameter per ovary, or ovarian volume >10 mL. "
            "The updated 2018 threshold recommends ≥20 follicles per ovary when using "
            "newer ultrasound technology."
        ),
    },
    # ── ESHRE Guidelines ───────────────────────────────────────────────
    {
        "id": "eshre_2018",
        "title": "International Evidence-Based Guideline for PCOS Assessment and Management",
        "year": "2018",
        "source": "ESHRE / Monash University",
        "document": (
            "The 2018 international evidence-based guideline for PCOS recommends a "
            "standardised diagnostic approach: Rotterdam criteria remain the gold standard. "
            "Anti-Müllerian Hormone (AMH) may be used as an adjunct or alternative to "
            "ultrasound for identifying polycystic ovarian morphology in adults, with "
            "levels ≥4.7 ng/mL suggestive of PCOS. The guideline emphasises screening for "
            "metabolic risk (glucose tolerance, lipid profile, blood pressure) at diagnosis "
            "and every 1-3 years. Cardiovascular risk assessment is recommended, as PCOS "
            "women have a 2-fold increased risk of coronary artery events. Lifestyle "
            "management (diet, exercise, behavioural strategies) is first-line therapy."
        ),
    },
    {
        "id": "eshre_2023_update",
        "title": "ESHRE 2023 Updated PCOS Guideline Recommendations",
        "year": "2023",
        "source": "Human Reproduction",
        "document": (
            "The 2023 update to the international PCOS guideline refines diagnostic "
            "thresholds and management. Key updates: AMH thresholds validated across "
            "populations (≥4.7 ng/mL in adults); the guideline now conditionally recommends "
            "AMH as a replacement for ultrasound in experienced settings. New emphasis on "
            "mental health screening — anxiety and depression affect up to 40% of PCOS patients. "
            "Metformin is recommended for metabolic features alongside or as an addition to "
            "combined oral contraceptive pills (COCPs). GLP-1 receptor agonists are emerging "
            "as therapeutic options. Sleep apnoea screening recommended for women with "
            "BMI >30 kg/m²."
        ),
    },
    {
        "id": "aes_criteria",
        "title": "Androgen Excess Society Position on PCOS Diagnostic Criteria",
        "year": "2006",
        "source": "Journal of Clinical Endocrinology & Metabolism",
        "document": (
            "The Androgen Excess Society (AES) proposed that hyperandrogenism (clinical or "
            "biochemical) should be a mandatory criterion for PCOS diagnosis, along with "
            "ovarian dysfunction (oligo-anovulation and/or polycystic ovaries). This excludes "
            "Rotterdam Phenotype D (oligo-anovulation + polycystic ovaries without "
            "hyperandrogenism). The AES argues that androgen excess is central to PCOS "
            "pathophysiology and without it, the diagnosis may include conditions unrelated "
            "to PCOS. Free testosterone or free androgen index (FAI) are preferred over "
            "total testosterone for biochemical assessment, as they better reflect "
            "biologically active androgen levels."
        ),
    },
    # ── Hormonal Markers ───────────────────────────────────────────────
    {
        "id": "testosterone_pcos",
        "title": "Testosterone Measurement in PCOS Diagnosis",
        "year": "2019",
        "source": "Clinical Chemistry",
        "document": (
            "Testosterone is the key biochemical marker for hyperandrogenism in PCOS. "
            "Total testosterone levels above the upper normal limit for the assay (typically "
            ">50-70 ng/dL in women) suggest biochemical hyperandrogenism. However, total "
            "testosterone can be normal in up to 50% of hyperandrogenic PCOS patients due "
            "to variations in sex hormone-binding globulin (SHBG). Free testosterone or "
            "calculated free androgen index (FAI = total testosterone × 100 / SHBG) are "
            "more sensitive. SHBG is typically low in PCOS due to hyperinsulinemia, "
            "increasing free testosterone fraction. LC-MS/MS is the gold standard assay."
        ),
    },
    {
        "id": "amh_pcos",
        "title": "Anti-Müllerian Hormone as a Diagnostic Marker for PCOS",
        "year": "2020",
        "source": "Reproductive BioMedicine Online",
        "document": (
            "Anti-Müllerian Hormone (AMH) is produced by granulosa cells of small antral "
            "follicles. In PCOS, the excess of small follicles leads to elevated AMH levels, "
            "typically 2-4 times higher than in non-PCOS women. AMH levels ≥4.7 ng/mL in "
            "adults have sensitivity of 82% and specificity of 79% for PCOS diagnosis. AMH "
            "correlates with antral follicle count (AFC), LH levels, and testosterone. It "
            "is not affected by menstrual cycle phase, making it a convenient diagnostic "
            "tool. Age-specific thresholds may improve accuracy: younger women (18-25) "
            "naturally have higher AMH, requiring adjusted cutoffs. AMH declines with age "
            "and may normalise in PCOS women over 35."
        ),
    },
    {
        "id": "lh_fsh_ratio",
        "title": "LH/FSH Ratio in PCOS: Clinical Significance and Limitations",
        "year": "2017",
        "source": "Gynecological Endocrinology",
        "document": (
            "An elevated LH/FSH ratio has historically been considered a hallmark of PCOS. "
            "Normal LH/FSH ratio is approximately 1:1 during the early follicular phase. "
            "In PCOS, LH is often disproportionately elevated, resulting in ratios of 2:1 "
            "or higher in 35-95% of patients depending on BMI and phenotype. The elevated "
            "LH drives theca cell androgen production, contributing to hyperandrogenism. "
            "However, the LH/FSH ratio is no longer a formal diagnostic criterion because "
            "it lacks sensitivity in obese PCOS patients (obesity suppresses LH pulsatility). "
            "It remains a useful supportive finding, especially in lean PCOS patients where "
            "ratios >2 are found in up to 60% of cases. FSH is typically normal or low in PCOS."
        ),
    },
    {
        "id": "tsh_thyroid",
        "title": "Thyroid Function and PCOS: Differential Diagnosis",
        "year": "2018",
        "source": "Endocrine Reviews",
        "document": (
            "Thyroid dysfunction must be excluded before diagnosing PCOS, as hypothyroidism "
            "can mimic PCOS symptoms including menstrual irregularity, weight gain, and "
            "anovulation. TSH should be measured in all women presenting with suspected PCOS. "
            "Normal TSH range is 0.4-4.0 mIU/L. Subclinical hypothyroidism (TSH 4-10 mIU/L) "
            "is more prevalent in PCOS (up to 25% vs 5-10% in general population). "
            "Hashimoto's thyroiditis shares an autoimmune overlap with PCOS. Elevated TSH "
            "increases prolactin via TRH stimulation, which can cause oligomenorrhea. "
            "Treatment of hypothyroidism may restore menstrual regularity and improve "
            "metabolic parameters in women with coexisting PCOS."
        ),
    },
    {
        "id": "prolactin_pcos",
        "title": "Prolactin Levels in PCOS and Differential Diagnosis",
        "year": "2015",
        "source": "Fertility and Sterility",
        "document": (
            "Mildly elevated prolactin (20-30 ng/mL) is found in up to 30% of PCOS patients. "
            "Prolactin >100 ng/mL should prompt investigation for prolactinoma. Normal "
            "prolactin range in non-pregnant women is 3-30 ng/mL. In PCOS, the mild "
            "hyperprolactinemia is thought to result from increased pituitary sensitivity "
            "to estrogen (from peripheral aromatisation of androgens) and altered dopaminergic "
            "tone. Exclusion of hyperprolactinemia as a cause of menstrual irregularity is "
            "required before confirming PCOS diagnosis. Prolactin levels >50 ng/mL usually "
            "indicate pathology other than PCOS (prolactinoma, medications, hypothyroidism)."
        ),
    },
    # ── Insulin Resistance & Metabolic ─────────────────────────────────
    {
        "id": "insulin_resistance",
        "title": "Insulin Resistance in PCOS: Mechanisms and Clinical Assessment",
        "year": "2019",
        "source": "Diabetes Care",
        "document": (
            "Insulin resistance (IR) affects 50-75% of PCOS women, independent of BMI. "
            "In PCOS, IR is tissue-selective: skeletal muscle and adipose tissue are resistant "
            "while the ovary remains insulin-sensitive, causing hyperinsulinemia to drive "
            "ovarian androgen production. HOMA-IR (fasting insulin × fasting glucose / 405) "
            "values >2.5 suggest insulin resistance. Fasting glucose >100 mg/dL or 2-hour "
            "glucose >140 mg/dL (OGTT) indicates impaired glucose metabolism. PCOS women "
            "have a 5-8 fold increased risk of type 2 diabetes. Acanthosis nigricans "
            "(skin darkening) is a clinical marker of insulin resistance. Metformin reduces "
            "fasting insulin by 30-40% and may restore ovulation in 30-50% of anovulatory "
            "PCOS patients."
        ),
    },
    {
        "id": "homa_ir_biomarker",
        "title": "HOMA-IR as a Biomarker in PCOS Metabolic Assessment",
        "year": "2021",
        "source": "Journal of Endocrinological Investigation",
        "document": (
            "HOMA-IR (Homeostatic Model Assessment of Insulin Resistance) is the most widely "
            "used clinical marker for insulin resistance. In PCOS, HOMA-IR >2.5 is considered "
            "abnormal. Mean HOMA-IR in PCOS populations is 3.0-4.5 compared to 1.5-2.0 in "
            "controls. Elevated fasting blood glucose (RBS >100 mg/dL) combined with high "
            "HOMA-IR strongly predicts metabolic syndrome in PCOS. Waist-to-hip ratio >0.85 "
            "indicates central adiposity and correlates with insulin resistance severity. "
            "BMI >25 kg/m² amplifies metabolic risk in PCOS. Annual screening with OGTT "
            "or HbA1c is recommended for all PCOS women regardless of BMI."
        ),
    },
    # ── Ovarian Morphology ─────────────────────────────────────────────
    {
        "id": "ovarian_morphology",
        "title": "Ultrasound Criteria for Polycystic Ovarian Morphology",
        "year": "2014",
        "source": "Ultrasound in Obstetrics & Gynecology",
        "document": (
            "Polycystic ovarian morphology (PCOM) on transvaginal ultrasound is defined by "
            "the presence of ≥12 follicles measuring 2-9 mm in diameter in at least one "
            "ovary, or an ovarian volume exceeding 10 mL. With improved ultrasound technology, "
            "the 2018 guideline updated the threshold to ≥20 follicles per ovary. Follicle "
            "count is typically performed in the early follicular phase (cycle days 2-5). "
            "Bilateral involvement is common but not required. The 'string of pearls' "
            "pattern (follicles arranged peripherally) is characteristic but not diagnostic. "
            "Ovarian volume is calculated using the simplified ellipsoid formula: "
            "0.5 × length × width × thickness. PCOM alone does not diagnose PCOS."
        ),
    },
    {
        "id": "follicle_count",
        "title": "Antral Follicle Count and Ovarian Reserve in PCOS",
        "year": "2020",
        "source": "Human Reproduction Update",
        "document": (
            "Antral follicle count (AFC) is the total number of follicles 2-9 mm visible "
            "on ultrasound. In PCOS, AFC is typically elevated (>20 total across both ovaries) "
            "due to arrested follicular development. The excess small follicles produce "
            "elevated AMH and contribute to the hyperandrogenic state. AFC correlates strongly "
            "with AMH (r=0.6-0.8). Follicle asymmetry (difference between left and right "
            "ovary count) may indicate unilateral dominance. Mean follicle size in PCOS "
            "is smaller (4-8 mm) compared to normal dominant follicle development (18-24 mm). "
            "Follicle count naturally declines with age, which must be considered when "
            "interpreting results in women over 35."
        ),
    },
    # ── Menstrual & Ovulatory ──────────────────────────────────────────
    {
        "id": "menstrual_irregularity",
        "title": "Menstrual Cycle Irregularity as a PCOS Diagnostic Criterion",
        "year": "2018",
        "source": "Best Practice & Research Clinical Obstetrics & Gynaecology",
        "document": (
            "Menstrual irregularity is one of the three Rotterdam diagnostic criteria. "
            "Oligomenorrhea is defined as cycles >35 days or <8 cycles/year. Amenorrhea "
            "is absence of menstruation for ≥3 consecutive months. In PCOS, 75-85% of "
            "women present with oligo- or amenorrhea. The irregularity stems from chronic "
            "anovulation due to the hyperandrogenic-hyperinsulinemic milieu disrupting "
            "follicular maturation. Cycle length >45 days is highly suggestive of anovulation. "
            "Progesterone measured mid-luteal phase <3 ng/mL confirms anovulation. "
            "Adolescents (within 2 years of menarche) may have physiological irregularity, "
            "requiring longer observation before diagnosing PCOS."
        ),
    },
    {
        "id": "anovulation_mechanisms",
        "title": "Mechanisms of Anovulation in PCOS",
        "year": "2016",
        "source": "Nature Reviews Endocrinology",
        "document": (
            "Anovulation in PCOS results from a self-perpetuating cycle: elevated LH "
            "stimulates theca cells to overproduce androgens, which are partially converted "
            "to estrogens via aromatase in granulosa cells and adipose tissue. The tonic "
            "(non-cyclic) estrogen exposure disrupts the GnRH pulse generator, maintaining "
            "elevated LH and relatively suppressed FSH. Without adequate FSH, follicles "
            "arrest at the antral stage (2-9 mm) and no dominant follicle emerges. "
            "Insulin amplifies this by: (1) enhancing ovarian androgen production, "
            "(2) reducing hepatic SHBG synthesis, increasing free androgens, and "
            "(3) amplifying LH-stimulated androgen secretion. Breaking this cycle "
            "requires addressing both insulin and androgen excess."
        ),
    },
    # ── Metabolic & Cardiovascular ─────────────────────────────────────
    {
        "id": "metabolic_syndrome",
        "title": "BMI and Metabolic Syndrome in PCOS",
        "year": "2020",
        "source": "Metabolism: Clinical and Experimental",
        "document": (
            "Metabolic syndrome (MetS) affects 30-45% of PCOS women vs 10-15% of age-matched "
            "controls. MetS criteria include: waist circumference >88 cm (women), triglycerides "
            ">150 mg/dL, HDL <50 mg/dL, BP >130/85 mmHg, and fasting glucose >100 mg/dL — "
            "3 of 5 criteria define MetS. BMI is the strongest modifier of metabolic risk in "
            "PCOS: obese PCOS (BMI >30) have 60% prevalence of MetS vs 10% in lean PCOS "
            "(BMI <25). Central adiposity (waist:hip ratio >0.85) is more predictive than "
            "BMI alone. Even lean PCOS women show higher visceral adiposity and worse "
            "insulin sensitivity compared to BMI-matched controls."
        ),
    },
    {
        "id": "cardiovascular_risk",
        "title": "Cardiovascular Risk Assessment in PCOS",
        "year": "2021",
        "source": "European Heart Journal",
        "document": (
            "PCOS is associated with a 2-fold increased risk of coronary artery events and "
            "a 1.5-fold increased stroke risk. Risk factors cluster in PCOS: dyslipidaemia "
            "(elevated LDL, low HDL, high triglycerides), hypertension, insulin resistance, "
            "and chronic low-grade inflammation (elevated CRP, IL-6). Systolic BP >130 mmHg "
            "or diastolic >85 mmHg should trigger lifestyle modification and monitoring. "
            "All PCOS women should be screened with a fasting lipid panel at diagnosis "
            "and every 1-2 years. Framingham risk score or similar calculators may "
            "underestimate risk in young PCOS women due to age-based weighting."
        ),
    },
    # ── Epidemiology ───────────────────────────────────────────────────
    {
        "id": "pcos_epidemiology",
        "title": "Global Epidemiology of PCOS",
        "year": "2020",
        "source": "The Lancet Diabetes & Endocrinology",
        "document": (
            "PCOS affects 8-13% of reproductive-age women globally, making it the most common "
            "endocrine disorder in this population. Prevalence varies by diagnostic criteria: "
            "NIH criteria (6-9%), Rotterdam criteria (8-13%), AES criteria (10-15%). "
            "Prevalence is higher in South Asian (up to 20%), Middle Eastern (16%), and "
            "Indigenous Australian (26%) women. First-degree relatives of PCOS women have "
            "a 40-50% risk of developing the condition. Up to 70% of PCOS cases remain "
            "undiagnosed. Average time to diagnosis is over 2 years, with women typically "
            "seeing 3+ healthcare providers before receiving a diagnosis."
        ),
    },
    {
        "id": "adolescent_pcos",
        "title": "Diagnosing PCOS in Adolescents",
        "year": "2017",
        "source": "Pediatric Endocrinology Reviews",
        "document": (
            "PCOS diagnosis in adolescents requires caution as physiological features of "
            "puberty overlap with PCOS. All three Rotterdam criteria should ideally be met. "
            "Menstrual irregularity is common in the first 2 years post-menarche and should "
            "only be considered pathological if cycles remain >90 days at 1 year post-menarche, "
            ">60 days at >2 years post-menarche, or if primary amenorrhea persists at age 15. "
            "Ultrasound criteria for PCOM are unreliable in adolescents due to normal "
            "multi-follicular ovaries. AMH >5.0 ng/mL may be more useful. Acne alone is not "
            "diagnostic. Severe hirsutism (Ferriman-Gallwey ≥8) with biochemical "
            "hyperandrogenism is the strongest diagnostic indicator in adolescents."
        ),
    },
    # ── Dermatological ─────────────────────────────────────────────────
    {
        "id": "hirsutism_scoring",
        "title": "Hirsutism Assessment: Modified Ferriman-Gallwey Score",
        "year": "2018",
        "source": "Journal of the American Academy of Dermatology",
        "document": (
            "The modified Ferriman-Gallwey (mFG) score is the standard clinical tool for "
            "assessing hirsutism. It evaluates terminal hair growth in 9 body areas: upper "
            "lip, chin, chest, upper back, lower back, upper abdomen, lower abdomen, upper "
            "arms, and thighs. Each area is scored 0-4, giving a maximum score of 36. "
            "Hirsutism is defined as mFG ≥4-6 (varies by ethnicity). Score 8-15 indicates "
            "moderate hirsutism; >15 indicates severe hirsutism. In PCOS, 60-80% of women "
            "with biochemical hyperandrogenism have clinical hirsutism. Ethnic variation is "
            "significant: East Asian women may have lower thresholds. Hirsutism correlates "
            "with free testosterone and 5α-reductase activity."
        ),
    },
    {
        "id": "acne_skin",
        "title": "Acne and Dermatological Manifestations of Hyperandrogenism",
        "year": "2019",
        "source": "Dermatologic Therapy",
        "document": (
            "Acne in PCOS is driven by androgen-stimulated sebum production. It typically "
            "presents as inflammatory papules and pustules along the jawline, chin, and "
            "lower face ('hormonal pattern'). Persistent acne beyond age 25 or treatment-resistant "
            "acne should raise suspicion for hyperandrogenism. Androgenic alopecia (female "
            "pattern hair loss) affects 40-70% of PCOS women, presenting as diffuse thinning "
            "at the crown with preservation of the frontal hairline. Acanthosis nigricans — "
            "dark, velvety skin in body folds (neck, axillae, groin) — indicates insulin "
            "resistance and is found in 30-50% of obese PCOS women. Skin tags are also "
            "associated with insulin resistance."
        ),
    },
    # ── Nutritional ────────────────────────────────────────────────────
    {
        "id": "vitamin_d_pcos",
        "title": "Vitamin D Deficiency in PCOS",
        "year": "2020",
        "source": "Nutrients",
        "document": (
            "Vitamin D deficiency (<20 ng/mL) is prevalent in 67-85% of PCOS women, "
            "significantly higher than age-matched controls (35-45%). Low vitamin D "
            "correlates with insulin resistance (r=-0.35), higher BMI, elevated testosterone, "
            "and worse metabolic profiles. Vitamin D receptors are expressed in ovarian "
            "tissue, and deficiency may impair follicular development. Supplementation "
            "(2000-4000 IU/day) has shown modest improvement in insulin sensitivity "
            "(HOMA-IR reduction of 0.5-1.0) and menstrual regularity in some trials. "
            "Optimal 25(OH)D target in PCOS is >30 ng/mL. Vitamin D insufficiency "
            "(20-30 ng/mL) affects an additional 15-20% of PCOS women."
        ),
    },
    # ── Treatment ──────────────────────────────────────────────────────
    {
        "id": "lifestyle_treatment",
        "title": "Lifestyle Modification as First-Line Treatment for PCOS",
        "year": "2019",
        "source": "Obesity Reviews",
        "document": (
            "Lifestyle modification is recommended as first-line therapy for all PCOS women. "
            "A 5-10% weight loss in overweight/obese PCOS women improves insulin sensitivity "
            "by 30-40%, reduces testosterone by 20-30%, and restores ovulation in 55-80% of "
            "cases. Recommended exercise: 150 min/week moderate-intensity or 75 min/week "
            "vigorous activity, plus 2 sessions of resistance training. Anti-inflammatory "
            "dietary patterns (Mediterranean, DASH) show benefits. Caloric restriction "
            "(500-750 kcal/day deficit) is recommended for weight loss. Behavioural "
            "strategies including cognitive behavioural therapy improve adherence and "
            "address the higher rates of disordered eating in PCOS (prevalence 20-35%)."
        ),
    },
    {
        "id": "metformin_treatment",
        "title": "Metformin in PCOS: Evidence and Clinical Use",
        "year": "2022",
        "source": "Cochrane Database of Systematic Reviews",
        "document": (
            "Metformin reduces hepatic glucose production and improves peripheral insulin "
            "sensitivity. In PCOS, metformin (1500-2550 mg/day) reduces fasting insulin by "
            "30%, testosterone by 15-20%, and BMI by 1-2 units over 6 months. It restores "
            "menstrual cyclicity in 40-70% of oligomenorrheic women. Metformin is recommended "
            "for PCOS women with BMI ≥25 who have inadequate response to lifestyle changes, "
            "or as adjunct to clomiphene for ovulation induction. Side effects (GI) affect "
            "20-30% of patients; extended-release formulation improves tolerance. Combined "
            "with lifestyle intervention, metformin shows greater improvements in ovulation "
            "rates and metabolic parameters than either alone."
        ),
    },
    {
        "id": "cocp_treatment",
        "title": "Combined Oral Contraceptives for PCOS Management",
        "year": "2021",
        "source": "Human Reproduction Update",
        "document": (
            "Combined oral contraceptive pills (COCPs) are first-line pharmacotherapy for "
            "menstrual irregularity and hyperandrogenism in PCOS. COCPs work by: "
            "(1) suppressing LH and ovarian androgen production, (2) increasing SHBG "
            "(reducing free testosterone), (3) providing regular withdrawal bleeds, and "
            "(4) protecting the endometrium from unopposed estrogen. Low-dose formulations "
            "(20-35 mcg ethinylestradiol) are preferred. Anti-androgenic progestins "
            "(cyproterone acetate, drospirenone) provide additional benefit for hirsutism "
            "and acne. COCPs may worsen insulin resistance and should be combined with "
            "lifestyle changes. They are contraindicated in women with VTE risk, severe "
            "hypertension, or migraine with aura."
        ),
    },
    # ── AI / ML in PCOS ────────────────────────────────────────────────
    {
        "id": "ml_pcos_diagnosis",
        "title": "Machine Learning Approaches for PCOS Diagnosis",
        "year": "2023",
        "source": "Artificial Intelligence in Medicine",
        "document": (
            "Machine learning models for PCOS detection achieve AUROC 0.85-0.97 depending "
            "on dataset and features used. XGBoost and Random Forest consistently outperform "
            "logistic regression. Key predictive features across studies include: antral "
            "follicle count, LH/FSH ratio, BMI, menstrual cycle regularity, hirsutism score, "
            "and skin darkening (acanthosis nigricans). SHAP (SHapley Additive exPlanations) "
            "provides feature-level explanations crucial for clinical acceptance. Ensemble "
            "methods combining clinical, hormonal, and ultrasound features achieve the best "
            "performance. Limitations include dataset size, population-specific biases, and "
            "the need for prospective validation. AI tools should augment, not replace, "
            "clinical judgement."
        ),
    },
    # ── Mental Health ──────────────────────────────────────────────────
    {
        "id": "mental_health_pcos",
        "title": "Mental Health Burden in PCOS: Anxiety and Depression",
        "year": "2022",
        "source": "Psychoneuroendocrinology",
        "document": (
            "PCOS women have 3-fold higher rates of depression and 5-fold higher rates of "
            "anxiety compared to controls. Up to 40% screen positive for depressive symptoms "
            "and 34% for anxiety. Contributing factors include hyperandrogenism-related body "
            "image distress (hirsutism, acne, weight), infertility concerns, and metabolic "
            "comorbidities. The 2023 ESHRE guideline recommends routine screening using "
            "validated tools (PHQ-9, GAD-7). Weight stigma and delayed diagnosis contribute "
            "to psychological burden. Exercise improves mood (effect size d=0.4-0.6) and "
            "should be part of holistic PCOS management. Referral to mental health "
            "professionals is recommended when screening is positive."
        ),
    },
]

print(f"Defined {len(CLINICAL_PAPERS)} clinical papers")

Defined 29 clinical papers


## 2 — Initialise ChromaDB Collection

In [3]:
import chromadb

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Delete collection if it exists (for clean re-runs)
try:
    chroma_client.delete_collection(COLLECTION)
    print(f"Deleted existing collection '{COLLECTION}'")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION,
    metadata={"hnsw:space": "cosine"},
)
print(f"Created collection '{COLLECTION}' with cosine similarity")

Deleted existing collection 'clinical_papers'
Created collection 'clinical_papers' with cosine similarity


## 3 — Embed & Store Documents

Chroma's default embedding function (`all-MiniLM-L6-v2`) handles vectorisation automatically — no Ollama needed for indexing.

In [4]:
collection.add(
    ids=[p["id"] for p in CLINICAL_PAPERS],
    documents=[p["document"] for p in CLINICAL_PAPERS],
    metadatas=[
        {"title": p["title"], "year": p["year"], "source": p["source"]}
        for p in CLINICAL_PAPERS
    ],
)

print(f"Added {collection.count()} documents to '{COLLECTION}'")
assert collection.count() >= 25, f"Expected ≥25 papers, got {collection.count()}"

Added 29 documents to 'clinical_papers'


## 4 — Test Retrieval

Run several queries to verify that semantic search returns relevant results.

In [5]:
test_queries = [
    "elevated testosterone and hyperandrogenism",
    "Rotterdam criteria for PCOS diagnosis",
    "insulin resistance and metabolic syndrome",
    "follicle count and ovarian morphology",
    "LH FSH ratio hormonal imbalance",
    "weight loss and lifestyle intervention for PCOS",
    "vitamin D deficiency in polycystic ovary syndrome",
]

for q in test_queries:
    results = collection.query(query_texts=[q], n_results=3)
    print(f"\n🔍 Query: \"{q}\"")
    for i, (doc_id, title, dist) in enumerate(
        zip(results["ids"][0], results["metadatas"][0], results["distances"][0])
    ):
        print(f"   {i+1}. [{dist:.4f}] {title['title']}")


🔍 Query: "elevated testosterone and hyperandrogenism"
   1. [0.3633] Testosterone Measurement in PCOS Diagnosis
   2. [0.3638] Androgen Excess Society Position on PCOS Diagnostic Criteria
   3. [0.4875] Hirsutism Assessment: Modified Ferriman-Gallwey Score

🔍 Query: "Rotterdam criteria for PCOS diagnosis"
   1. [0.3209] PCOS Phenotypes Under the Rotterdam Criteria
   2. [0.3434] Global Epidemiology of PCOS
   3. [0.3449] Rotterdam ESHRE/ASRM-Sponsored PCOS Consensus Workshop 2003

🔍 Query: "insulin resistance and metabolic syndrome"
   1. [0.4264] Insulin Resistance in PCOS: Mechanisms and Clinical Assessment
   2. [0.4337] HOMA-IR as a Biomarker in PCOS Metabolic Assessment
   3. [0.4695] BMI and Metabolic Syndrome in PCOS

🔍 Query: "follicle count and ovarian morphology"
   1. [0.4139] Antral Follicle Count and Ovarian Reserve in PCOS
   2. [0.4511] Applying the Rotterdam Criteria in Clinical Practice
   3. [0.4639] Ultrasound Criteria for Polycystic Ovarian Morphology

🔍 Query: "LH

## 5 — Verify Collection Integrity

In [6]:
# Peek at a few stored documents
peek = collection.peek(limit=3)
for doc_id, doc, meta in zip(peek["ids"], peek["documents"], peek["metadatas"]):
    print(f"  {doc_id:30s} | {meta['title'][:50]:50s} | {len(doc)} chars")

print(f"\n✅ ChromaDB collection '{COLLECTION}' ready")
print(f"   Documents : {collection.count()}")
print(f"   Storage   : {CHROMA_DIR}")
print(f"   Embedding : all-MiniLM-L6-v2 (Chroma default)")

  rotterdam_2003                 | Rotterdam ESHRE/ASRM-Sponsored PCOS Consensus Work | 649 chars
  rotterdam_phenotypes           | PCOS Phenotypes Under the Rotterdam Criteria       | 630 chars
  rotterdam_application          | Applying the Rotterdam Criteria in Clinical Practi | 626 chars

✅ ChromaDB collection 'clinical_papers' ready
   Documents : 29
   Storage   : /Users/apple/Desktop/GradStudy/SYSEN5381/PCOSense/knowledge_base/chroma_db
   Embedding : all-MiniLM-L6-v2 (Chroma default)


## 6 — Quick RAG Integration Test

Test the full RAG pipeline (`src/rag_system.py`) if Ollama is running.

In [7]:
from src.rag_system import RAGSystem

rag = RAGSystem(chroma_dir=CHROMA_DIR)
print(f"RAG system loaded: {rag.paper_count()} papers\n")

# Retrieval-only test (no Ollama needed)
papers = rag.retrieve_papers("elevated testosterone and LH in women", n_results=3)
for p in papers:
    print(f"  [{p['distance']:.4f}] {p['metadata']['title']}")

# Full synthesis test (requires Ollama)
if rag.ollama.is_available():
    result = rag.synthesize_evidence("What are the Rotterdam criteria for PCOS?")
    print(f"\n📄 Synthesis:\n{result['clinical_summary'][:500]}")
else:
    print("\n⚠️  Ollama not running — skipping synthesis test.")
    print("   Start Ollama and pull llama3.2: ollama pull llama3.2")

RAG system loaded: 29 papers

  [0.3954] LH/FSH Ratio in PCOS: Clinical Significance and Limitations
  [0.4253] Anti-Müllerian Hormone as a Diagnostic Marker for PCOS
  [0.4581] Mechanisms of Anovulation in PCOS

📄 Synthesis:
**Clinical Summary: Rotterdam Criteria for Polycystic Ovary Syndrome (PCOS)**

The Rotterdam criteria, established in 2003, provide a comprehensive diagnostic framework for PCOS. The criteria require at least two of three features:

1. **Oligo-ovulation or anovulation**: Infrequent menstrual cycles (<9 days) or amenorrhea.
2. **Clinical and/or biochemical signs of hyperandrogenism**: Elevated serum testosterone levels (>20 nmol/L), hirsutism, acne, or male pattern baldness.
3. **Polycystic ovari


---

**Knowledge base is ready!**  
Agent 2 (`ClinicalEvidenceRetriever`) will use `src/rag_system.py` to search this collection at runtime.